In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# =========================
# Cell 1
# Setup
# =========================

from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import os

auth.authenticate_user()
client = bigquery.Client(project="mimic-rs")

BASE_PATH = "/content/drive/MyDrive/Plan A/data/eicu"
os.makedirs(BASE_PATH, exist_ok=True)

print("Setup complete")

Setup complete


In [5]:
# =========================
# Cell 2
# eICU cohort
# =========================

query =query = """
SELECT
  p.patientunitstayid AS stay_id,
  p.patienthealthsystemstayid AS hadm_id,
  p.uniquepid AS subject_id,
  p.gender,

  -- FIX AGE
  CASE
    WHEN p.age = '> 89' THEN 90
    ELSE SAFE_CAST(p.age AS INT64)
  END AS age,

  p.unitadmittime24 AS intime,
  p.unitdischargetime24 AS outtime,
  p.hospitaldischargestatus

FROM `physionet-data.eicu_crd.patient` p
WHERE
  (
    CASE
      WHEN p.age = '> 89' THEN 90
      ELSE SAFE_CAST(p.age AS INT64)
    END
  ) >= 18
"""

df = client.query(query).to_dataframe()

# mortality mapping
df["mortality"] = df["hospitaldischargestatus"].map({
    "Expired": 1,
    "Alive": 0
})

df = df.dropna(subset=["mortality"])

print("Shape:", df.shape)
df.head()

Shape: (198490, 9)


,stay_id,hadm_id,subject_id,gender,age,intime,outtime,hospitaldischargestatus,mortality
0,408811,349187,004-42773,,31,07:56:00,15:45:00,Alive,0.0
1,1333762,1014888,013-27781,,27,21:58:00,22:10:00,Alive,0.0
2,2792401,2253390,027-116986,,23,11:13:00,23:14:00,Alive,0.0
3,1457892,1116627,015-93748,Male,22,21:18:00,05:30:00,Alive,0.0
4,2685936,2154643,027-99100,Male,20,21:21:00,00:53:00,Alive,0.0


In [6]:
df = client.query(query).to_dataframe()

df["mortality"] = df["hospitaldischargestatus"].map({
    "Expired": 1,
    "Alive": 0
})

df = df.dropna(subset=["mortality"])

print("Shape:", df.shape)
print("Mortality rate:", df["mortality"].mean())

Shape: (198490, 9)
Mortality rate: 0.09059398458360622


In [10]:
# =========================
# Cell 3
# eICU Heart Rate (first 24h)
# =========================

query = query = query = """
WITH cohort AS (
  SELECT
    p.patientunitstayid AS stay_id
  FROM `physionet-data.eicu_crd.patient` p
  WHERE
    (
      CASE
        WHEN p.age = '> 89' THEN 90
        ELSE SAFE_CAST(p.age AS INT64)
      END
    ) >= 18
),

hr_events AS (
  SELECT
    c.stay_id,
    SAFE_CAST(nc.nursingchartvalue AS FLOAT64) AS hr
  FROM cohort c
  JOIN `physionet-data.eicu_crd.nursecharting` nc
    ON c.stay_id = nc.patientunitstayid
  WHERE LOWER(nc.nursingchartcelltypevallabel) = 'heart rate'
    AND SAFE_CAST(nc.nursingchartvalue AS FLOAT64) BETWEEN 30 AND 180
    AND nc.nursingchartoffset BETWEEN 0 AND 1440
),

-- 🔥 KEY FIX: remove duplicate bias via aggregation at event level
hr_clean AS (
  SELECT DISTINCT
    stay_id,
    hr
  FROM hr_events
)

SELECT
  c.stay_id,
  COUNT(h.hr) AS hr_n,
  MIN(h.hr) AS hr_min,
  MAX(h.hr) AS hr_max,
  AVG(h.hr) AS hr_mean
FROM cohort c
LEFT JOIN hr_clean h
  ON c.stay_id = h.stay_id
GROUP BY c.stay_id
ORDER BY c.stay_id
"""

hr_df = client.query(query).to_dataframe()

print("Shape:", hr_df.shape)
hr_df.head()

Shape: (200234, 5)


,stay_id,hr_n,hr_min,hr_max,hr_mean
0,141168,11,70.0,140.0,117.454545
1,141178,2,86.0,88.0,87.000000
2,141179,11,78.0,102.0,89.454545
3,141194,15,72.0,108.0,87.066667
4,141196,6,73.0,106.0,88.333333


In [9]:
query = """
SELECT
  nursingchartcelltypevallabel,
  COUNT(*) as cnt
FROM `physionet-data.eicu_crd.nursecharting`
WHERE LOWER(nursingchartcelltypevallabel) LIKE '%heart%'
GROUP BY nursingchartcelltypevallabel
ORDER BY cnt DESC
LIMIT 20
"""

audit_df = client.query(query).to_dataframe()
audit_df

,nursingchartcelltypevallabel,cnt
0,Heart Rate,17565063


In [11]:
print(hr_df[['hr_min','hr_max','hr_mean']].describe())

print("Missing HR:", (hr_df['hr_n'] == 0).sum())

              hr_min         hr_max        hr_mean
count  181233.000000  181233.000000  181233.000000
mean       72.174345     101.367985      85.427640
std        16.122222      21.633195      16.471746
min        30.000000      30.000000      30.000000
25%        61.000000      86.000000      73.500000
50%        71.000000      99.000000      84.066667
75%        82.000000     115.000000      96.090909
max       175.000000     180.000000     175.000000
Missing HR: 19001


In [22]:
# =========================
# Cell 5
# eICU Vitals (first 24h) — fixed
# =========================

# =========================
# Cell 5 (FINAL FIXED)
# eICU Vitals
# =========================

query = """
WITH cohort AS (
  SELECT
    p.patientunitstayid AS stay_id
  FROM `physionet-data.eicu_crd.patient` p
  WHERE
    (
      CASE
        WHEN p.age = '> 89' THEN 90
        ELSE SAFE_CAST(p.age AS INT64)
      END
    ) >= 18
),

events AS (
  SELECT
    c.stay_id,
    LOWER(nc.nursingchartcelltypevallabel) AS label,
    SAFE_CAST(nc.nursingchartvalue AS FLOAT64) AS val_num
  FROM cohort c
  JOIN `physionet-data.eicu_crd.nursecharting` nc
    ON c.stay_id = nc.patientunitstayid
  WHERE nc.nursingchartoffset BETWEEN 0 AND 1440
),

parsed AS (
  SELECT
    stay_id,

    -- MAP ONLY (correct)
    CASE
      WHEN label = 'non-invasive bp' AND val_num BETWEEN 30 AND 150
      THEN val_num
    END AS map_val,

    CASE
      WHEN label = 'respiratory rate' AND val_num BETWEEN 5 AND 60
      THEN val_num
    END AS rr_val,

    CASE
      WHEN label = 'o2 saturation' AND val_num BETWEEN 50 AND 100
      THEN val_num
    END AS spo2_val

  FROM events
),

clean AS (
  SELECT DISTINCT
    stay_id,
    map_val,
    rr_val,
    spo2_val
  FROM parsed
)

SELECT
  c.stay_id,

  COUNT(f.map_val) AS map_n,
  MIN(f.map_val) AS map_min,
  MAX(f.map_val) AS map_max,
  AVG(f.map_val) AS map_mean,

  COUNT(f.rr_val) AS rr_n,
  MIN(f.rr_val) AS rr_min,
  MAX(f.rr_val) AS rr_max,
  AVG(f.rr_val) AS rr_mean,

  COUNT(f.spo2_val) AS spo2_n,
  MIN(f.spo2_val) AS spo2_min,
  MAX(f.spo2_val) AS spo2_max,
  AVG(f.spo2_val) AS spo2_mean

FROM cohort c
LEFT JOIN clean f
  ON c.stay_id = f.stay_id
GROUP BY c.stay_id
ORDER BY c.stay_id
"""

vitals_df = client.query(query).to_dataframe()

print("Shape:", vitals_df.shape)
display(vitals_df.head())

Shape: (200234, 13)


,stay_id,map_n,map_min,map_max,map_mean,rr_n,rr_min,rr_max,rr_mean,spo2_n,spo2_min,spo2_max,spo2_mean
0,141168,12,59.0,111.0,84.833333,6,16.0,30.0,24.333333,9,68.0,99.0,89.777778
1,141178,3,62.0,100.0,75.333333,2,16.0,20.0,18.000000,2,92.0,99.0,95.500000
2,141179,23,53.0,126.0,87.565217,4,16.0,20.0,17.750000,5,95.0,100.0,97.600000
3,141194,54,37.0,115.0,68.851852,17,10.0,34.0,22.823529,7,93.0,100.0,96.857143
4,141196,17,61.0,130.0,95.529412,4,19.0,28.0,22.750000,4,88.0,97.0,93.250000


In [25]:
# =========================
# Correct validation
# =========================

print(vitals_df[['map_mean','rr_mean','spo2_mean']].describe())

print({
    'map_missing': (vitals_df['map_n'] == 0).sum(),
    'rr_missing': (vitals_df['rr_n'] == 0).sum(),
    'spo2_missing': (vitals_df['spo2_n'] == 0).sum(),
})

            map_mean        rr_mean      spo2_mean
count  178093.000000  174194.000000  162830.000000
mean       89.145141      19.936760      95.944509
std        10.658307       4.398855       2.674010
min        30.000000       5.000000      50.000000
25%        81.873016      17.000000      94.800000
50%        89.071429      19.166667      96.250000
75%        96.491525      22.200000      97.500000
max       149.000000      58.000000     100.000000
{'map_missing': np.int64(22141), 'rr_missing': np.int64(26040), 'spo2_missing': np.int64(37404)}


In [20]:
print(vitals_df[['sbp_mean','dbp_mean','map_mean','rr_mean','spo2_mean']].describe())

print({
    'sbp_missing': (vitals_df['sbp_n'] == 0).sum(),
    'dbp_missing': (vitals_df['dbp_n'] == 0).sum(),
    'map_missing': (vitals_df['map_n'] == 0).sum(),
    'rr_missing': (vitals_df['rr_n'] == 0).sum(),
    'spo2_missing': (vitals_df['spo2_n'] == 0).sum(),
})

       sbp_mean  dbp_mean  map_mean        rr_mean      spo2_mean
count       0.0       0.0       0.0  174194.000000  162830.000000
mean        NaN       NaN       NaN      19.451960      96.702590
std         NaN       NaN       NaN       4.372583       2.539594
min         NaN       NaN       NaN       5.000000      50.000000
25%         NaN       NaN       NaN      16.476190      95.466667
50%         NaN       NaN       NaN      18.700000      97.000000
75%         NaN       NaN       NaN      21.684211      98.400000
max         NaN       NaN       NaN      58.000000     100.000000
{'sbp_missing': np.int64(200234), 'dbp_missing': np.int64(200234), 'map_missing': np.int64(200234), 'rr_missing': np.int64(26040), 'spo2_missing': np.int64(37404)}


In [31]:
# =========================
# Cell 7
# eICU Labs (first 24h) — FINAL
# =========================

query = """
WITH cohort AS (
  SELECT
    p.patientunitstayid AS stay_id
  FROM `physionet-data.eicu_crd.patient` p
  WHERE
    (
      CASE
        WHEN p.age = '> 89' THEN 90
        ELSE SAFE_CAST(p.age AS INT64)
      END
    ) >= 18
),

lab_events AS (
  SELECT
    c.stay_id,
    LOWER(l.labname) AS lab,
    SAFE_CAST(l.labresult AS FLOAT64) AS val,
    l.labresultoffset
  FROM cohort c
  JOIN `physionet-data.eicu_crd.lab` l
    ON c.stay_id = l.patientunitstayid
  WHERE l.labresultoffset BETWEEN 0 AND 1440
),

filtered AS (
  SELECT
    stay_id,

    CASE WHEN lab = 'creatinine' AND val BETWEEN 0.1 AND 20 THEN val END AS creat,
    CASE WHEN lab = 'lactate' AND val BETWEEN 0.2 AND 20 THEN val END AS lact,

    CASE WHEN lab = 'wbc x 1000' AND val BETWEEN 0.1 AND 200 THEN val END AS wbc,
    CASE WHEN lab = 'hgb' AND val BETWEEN 2 AND 25 THEN val END AS hgb,
    CASE WHEN lab = 'platelets x 1000' AND val BETWEEN 5 AND 1000 THEN val END AS plt,

    CASE WHEN lab = 'sodium' AND val BETWEEN 110 AND 180 THEN val END AS sodium

  FROM lab_events
),

clean AS (
  SELECT DISTINCT
    stay_id,
    creat,
    lact,
    wbc,
    hgb,
    plt,
    sodium
  FROM filtered
)

SELECT
  c.stay_id,

  COUNT(f.creat) AS creat_n,
  AVG(f.creat) AS creat_mean,

  COUNT(f.lact) AS lact_n,
  AVG(f.lact) AS lact_mean,

  COUNT(f.wbc) AS wbc_n,
  AVG(f.wbc) AS wbc_mean,

  COUNT(f.hgb) AS hgb_n,
  AVG(f.hgb) AS hgb_mean,

  COUNT(f.plt) AS plt_n,
  AVG(f.plt) AS plt_mean,

  COUNT(f.sodium) AS sodium_n,
  AVG(f.sodium) AS sodium_mean

FROM cohort c
LEFT JOIN clean f
  ON c.stay_id = f.stay_id

GROUP BY c.stay_id
ORDER BY c.stay_id
"""

labs_df = client.query(query).to_dataframe()

print("Shape:", labs_df.shape)
display(labs_df.head())

Shape: (200234, 13)


,stay_id,creat_n,creat_mean,lact_n,lact_mean,wbc_n,wbc_mean,hgb_n,hgb_mean,plt_n,plt_mean,sodium_n,sodium_mean
0,141168,2,2.125,0,NaN,2,12.25,2,13.15,2,211.0,1,139.0
1,141178,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN
2,141179,1,0.700,0,NaN,0,NaN,0,NaN,0,NaN,2,143.0
3,141194,3,2.340,2,1.15,1,14.10,1,8.90,1,233.0,3,135.0
4,141196,1,0.800,0,NaN,1,26.20,1,10.30,1,453.0,1,135.0


In [27]:
print(labs_df[['creat_mean','lact_mean','wbc_mean','hgb_mean','plt_mean','sodium_mean']].describe())

print({
    'creat_missing': (labs_df['creat_n'] == 0).sum(),
    'lact_missing': (labs_df['lact_n'] == 0).sum(),
    'wbc_missing': (labs_df['wbc_n'] == 0).sum(),
    'hgb_missing': (labs_df['hgb_n'] == 0).sum(),
    'plt_missing': (labs_df['plt_n'] == 0).sum(),
    'sodium_missing': (labs_df['sodium_n'] == 0).sum(),
})

          creat_mean     lact_mean  wbc_mean  hgb_mean  plt_mean  \
count  169436.000000  43769.000000       0.0       0.0       0.0   
mean        1.480324      2.420234       NaN       NaN       NaN   
std         1.562230      2.398161       NaN       NaN       NaN   
min         0.100000      0.200000       NaN       NaN       NaN   
25%         0.730000      1.100000       NaN       NaN       NaN   
50%         0.982500      1.700000       NaN       NaN       NaN   
75%         1.500000      2.700000       NaN       NaN       NaN   
max        20.000000     20.000000       NaN       NaN       NaN   

         sodium_mean  
count  169627.000000  
mean      138.479775  
std         4.899314  
min       110.000000  
25%       136.000000  
50%       139.000000  
75%       141.000000  
max       179.000000  
{'creat_missing': np.int64(30798), 'lact_missing': np.int64(156465), 'wbc_missing': np.int64(200234), 'hgb_missing': np.int64(200234), 'plt_missing': np.int64(200234), 'sodium_miss

In [32]:
# =========================
# Cell 8
# Validation
# =========================

print(labs_df[['creat_mean','lact_mean','wbc_mean','hgb_mean','plt_mean','sodium_mean']].describe())

print({
    'creat_missing': (labs_df['creat_n'] == 0).sum(),
    'lact_missing': (labs_df['lact_n'] == 0).sum(),
    'wbc_missing': (labs_df['wbc_n'] == 0).sum(),
    'hgb_missing': (labs_df['hgb_n'] == 0).sum(),
    'plt_missing': (labs_df['plt_n'] == 0).sum(),
    'sodium_missing': (labs_df['sodium_n'] == 0).sum(),
})

          creat_mean     lact_mean       wbc_mean       hgb_mean  \
count  169436.000000  43769.000000  162333.000000  164289.000000   
mean        1.480324      2.420234      11.792962      10.962371   
std         1.562230      2.398161       7.248609       2.208704   
min         0.100000      0.200000       0.100000       2.000000   
25%         0.730000      1.100000       7.600000       9.266667   
50%         0.982500      1.700000      10.400000      10.900000   
75%         1.500000      2.700000      14.200000      12.500000   
max        20.000000     20.000000     198.100000      22.700000   

            plt_mean    sodium_mean  
count  160438.000000  169627.000000  
mean      203.426230     138.479775  
std        94.908267       4.899314  
min         5.000000     110.000000  
25%       142.000000     136.000000  
50%       191.000000     139.000000  
75%       249.000000     141.000000  
max       998.000000     179.000000  
{'creat_missing': np.int64(30798), 'lact_miss

In [33]:
# =========================
# Save cohort
# =========================

save_path = "/content/drive/MyDrive/Plan A/data/eicu/eicu_cohort.csv"
df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /content/drive/MyDrive/Plan A/data/eicu/eicu_cohort.csv


In [34]:
save_path = "/content/drive/MyDrive/Plan A/data/eicu/eicu_vitals_24h.csv"
vitals_df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /content/drive/MyDrive/Plan A/data/eicu/eicu_vitals_24h.csv


In [35]:
save_path = "/content/drive/MyDrive/Plan A/data/eicu/eicu_labs_24h.csv"
labs_df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /content/drive/MyDrive/Plan A/data/eicu/eicu_labs_24h.csv


In [37]:
save_path = "/content/drive/MyDrive/Plan A/data/eicu/eicu_hr_24h.csv"
hr_df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /content/drive/MyDrive/Plan A/data/eicu/eicu_hr_24h.csv


In [38]:
import os

base = "/content/drive/MyDrive/Plan A/data/eicu"
print(os.listdir(base))

['eicu_cohort.csv', 'eicu_vitals_24h.csv', 'eicu_labs_24h.csv', 'eicu_hr_24h.csv']
